<div align="center">

# 🚕 Yandex Go Analytics

## 6 · Data Enrichment

*Join every trip with its driver's profile in a single, analysis-ready table.*

![layer](https://img.shields.io/badge/layer-gold-FFB300)
![operation](https://img.shields.io/badge/operation-LEFT%20JOIN-00ADD4)
![table](https://img.shields.io/badge/table-enriched_trips-2E7D32)

</div>

| ◀ Previous | 🏠 Home | Next ▶ |
|:--|:--:|--:|
| [5 · Aggregations & Optimizations](./5.%20Aggregations%20%26%20Optimizations.ipynb) | [Project Overview](./0.%20Project%20Overview.ipynb) | [7 · Refactor via Delta Live Tables](./7.%20Refactor%20via%20Delta%20Live%20Tables.ipynb) |

**Progress** &nbsp; ▰▰▰▰▰▰▱▱  **6 / 8**

---

## 🎯 Task requirements

- [x] Generate a **synthetic `driver_id`** for the taxi trips
- [x] Perform a **`LEFT JOIN`** between the trips and the driver profiles
- [x] Save the unified, enriched dataset as **`gold.enriched_trips`**

## 🧩 How it is solved

**Synthetic key.** The trip generator assigns every trip a `driver_id` drawn from a *deterministic* UUID pool
(fixed Faker seed). The driver generator uses the **same** pool, so trips and driver profiles share a join key
without any real-world identifiers.

```text
 silver.taxi  (stream, de-duplicated by trip id)
      │  driver_id
      ▼
   LEFT JOIN  ◀──────────  silver.drivers  (static snapshot, one row per driver id)
      │
      ▼
 gold.enriched_trips   =   all trip columns  +  driver_name · driver_car_number
                                                driver_experience · driver_rating [· driver_phone]
```

| Design choice | Why |
|:--|:--|
| `LEFT JOIN` | A trip must survive even when its driver profile is missing or was rejected by Silver |
| Stream ⨝ static | New trips flow in incrementally; the (small) driver dimension is read as a snapshot |
| `driver_` prefix | Keeps trip and driver columns unambiguous |
| `driver_phone` only if present | The `phone` column was rolled back in notebook 3, so it is added only if it exists in Silver |

## 1️⃣ Connect to Spark

In [1]:
from pyspark.sql import functions as F

from src.core.config import get_catalog, get_environment
from src.core.infra import DatabricksConnectionService

conn = DatabricksConnectionService(app_name="yandex-go-enrichment")
conn.connect()
spark = conn.spark

CATALOG = get_catalog()
print(f"✅ Connected | Spark {spark.version} | catalog = {CATALOG} | env = {get_environment()}")

2026-10-09 15:26:39.820 | INFO     | src.core.infra.connection.databricks_connector:connect:56 - SparkSession ready | app=yandex-go-enrichment | version=4.2.0 | runtime=serverless-connect


✅ Connected | Spark 4.2.0 | catalog = yandex_go_staging | env = staging


## 2️⃣ Build `gold.enriched_trips`

In [2]:
from src.core.etl.gold import EnrichedTripsGoldService

EnrichedTripsGoldService().run(spark)

enriched = spark.table(f"{CATALOG}.gold.enriched_trips")
print(f"gold.enriched_trips: {enriched.count():,} rows | {len(enriched.columns)} columns")
display(enriched.toPandas())

2026-10-09 15:26:39.831 | INFO     | src.core.etl.gold.trip_driver:run:34 - Reading stream from silver.taxi via checkpoints...
2026-10-09 15:27:11.115 | INFO     | src.core.etl.gold.trip_driver:run:73 - Enriched yandex_go_trips_pipeline successfully written!


gold.enriched_trips: 1,913 rows | 32 columns


,driver_id,id,user_id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,...,_rescued_data,_ingested_at,_source_file,payment_type_description,_processed_at,driver_name,driver_car_number,driver_experience,driver_rating,_calculated_at
0,ff9a1b80-5ced-4e2e-b17f-6920daaafe5c,35acb0fe-bcf9-4554-a86b-4ef673d9f9e1,43b9da13-ec85-4f37-bbc1-a987aff8754d,2,2026-10-03 00:32:30,2026-10-03 00:37:30,6,9.74,1,Y,...,None,2026-10-09 12:21:23.959,/Volumes/yandex_go_staging/raw_files/landing/p...,Other,2026-10-09 12:22:39.438,None,None,NaN,NaN,2026-10-09 12:26:43.568
1,f3a3c571-7476-4899-b5a3-adb3254a9493,1cc33e07-4ab0-4896-a952-d17640c53c5a,90b2b633-956b-4c0c-a849-9b926b5252e3,2,2026-10-03 03:26:14,2026-10-03 03:48:14,3,11.94,3,N,...,None,2026-10-09 12:21:23.959,/Volumes/yandex_go_staging/raw_files/landing/p...,Cash,2026-10-09 12:22:39.438,John Reyes,Y607CC,7.0,4.95,2026-10-09 12:26:43.568
2,467045a0-c0fe-45dd-8df9-564b2a2a146d,ec737933-8d5b-4844-b9ae-7a199b8f8207,eededb07-e623-4689-9d59-cd2a4eea04e7,1,2026-10-08 10:36:46,2026-10-08 10:46:46,3,16.47,5,Y,...,None,2026-10-09 12:21:23.959,/Volumes/yandex_go_staging/raw_files/landing/p...,Other,2026-10-09 12:22:39.438,None,None,NaN,NaN,2026-10-09 12:26:43.568
3,db427e48-2911-4759-a006-7d39d169a3f8,cdce2a55-b307-4208-98bd-53d2989d69ae,e9d68f23-b489-4070-b914-f8a8bea4ff31,1,2026-10-05 11:16:31,2026-10-05 11:41:31,5,22.24,5,Y,...,None,2026-10-09 12:21:23.959,/Volumes/yandex_go_staging/raw_files/landing/p...,Other,2026-10-09 12:22:39.438,Brandon Jenkins,O826HE,2.0,4.65,2026-10-09 12:26:43.568
4,fcc1b543-c49c-465b-96c2-703c4f353a13,8992de76-18e0-481c-8d43-14cbf15679f4,670acc5c-b321-4f21-8dd8-eb85b04d3376,2,2026-10-01 22:27:54,2026-10-01 22:32:54,3,25.75,1,Y,...,None,2026-10-09 12:21:23.959,/Volumes/yandex_go_staging/raw_files/landing/p...,Credit Card,2026-10-09 12:22:39.438,None,None,NaN,NaN,2026-10-09 12:26:43.568
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1908,6e149b07-7559-4c0e-8dbb-1cb464be22a3,2c2296f5-d5ee-4cfb-808c-8e8cc0ba6dfc,8a3c3b5e-801e-41da-85b1-ed25f1533ae8,2,2026-10-07 23:30:17,2026-10-08 00:11:17,5,12.57,1,N,...,None,2026-10-09 12:21:23.959,/Volumes/yandex_go_staging/raw_files/landing/p...,Other,2026-10-09 12:22:39.438,Shelby Perry,O252EK,15.0,4.69,2026-10-09 12:26:43.568
1909,cbe42623-4572-43d3-80b9-d827457af00a,e0afc3e8-d507-4833-b928-08697271fcbb,bf0d073d-821c-4336-9970-cf60ebff8d15,2,2026-10-02 05:26:20,2026-10-02 06:07:20,4,17.53,3,N,...,None,2026-10-09 12:21:23.959,/Volumes/yandex_go_staging/raw_files/landing/p...,Other,2026-10-09 12:22:39.438,None,None,NaN,NaN,2026-10-09 12:26:43.568
1910,8646b8ad-4f52-4577-a2cb-b9ab3819fe4d,742b9436-40f4-4b31-87de-353cde847c2e,402913ec-9ef2-493e-b0ac-7d7ba2f963a3,1,2026-10-09 03:12:26,2026-10-09 03:27:26,6,1.77,1,Y,...,None,2026-10-09 12:21:23.959,/Volumes/yandex_go_staging/raw_files/landing/p...,Cash,2026-10-09 12:22:39.438,Jennifer Reynolds,C352MO,8.0,4.82,2026-10-09 12:26:43.568
1911,e34d871b-c457-4a62-affb-6348b6f9706f,69193d04-aee0-476f-88f0-873695f24d2c,ce88cb2d-d4e8-4839-bc3e-058be0f3eab0,2,2026-10-01 20:53:24,2026-10-01 21:29:24,6,13.89,5,N,...,None,2026-10-09 12:21:23.959,/Volumes/yandex_go_staging/raw_files/landing/p...,Credit Card,2026-10-09 12:22:39.438,None,None,NaN,NaN,2026-10-09 12:26:43.568


## 3️⃣ Verify the join

In [3]:
def check(label: str, condition: bool) -> None:
    print(("✅" if condition else "❌"), label)
    assert condition, label

silver_trip_count = spark.table(f"{CATALOG}.silver.taxi").select("id").distinct().count()
enriched_count = enriched.count()

check(
    f"row-preserving: {enriched_count:,} enriched rows == {silver_trip_count:,} distinct Silver trips",
    enriched_count == silver_trip_count,
)
check("one row per trip id", enriched_count == enriched.select("id").distinct().count())
check(
    "driver columns are present",
    {"driver_name", "driver_car_number", "driver_experience", "driver_rating"} <= set(enriched.columns),
)

✅ row-preserving: 1,913 enriched rows == 1,913 distinct Silver trips
✅ one row per trip id
✅ driver columns are present


### 🎯 Match rate

Trips whose driver has no Silver profile keep `NULL` driver columns — the signature of a `LEFT JOIN`.

In [4]:
match = (
    enriched.agg(
        F.count("*").alias("total_trips"),
        F.sum(F.when(F.col("driver_name").isNotNull(), 1).otherwise(0)).alias("with_driver"),
        F.sum(F.when(F.col("driver_name").isNull(), 1).otherwise(0)).alias("without_driver"),
    )
    .withColumn("match_rate_pct", F.round(F.col("with_driver") / F.col("total_trips") * 100, 1))
)
display(match.toPandas())

assert match.first()["with_driver"] > 0, "No trip was matched with a driver profile"
print("✅ Trips are enriched with driver profiles")

,total_trips,with_driver,without_driver,match_rate_pct
0,1913,725,1188,37.9


✅ Trips are enriched with driver profiles


### 🏆 What the enriched table unlocks

Revenue per driver — impossible without the join.

In [5]:
top_drivers = (
    enriched.filter(F.col("driver_name").isNotNull())
    .groupBy("driver_id", "driver_name", "driver_experience", "driver_rating")
    .agg(
        F.count("*").alias("trips"),
        F.round(F.sum("fare_amount"), 2).alias("revenue"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
    )
    .orderBy(F.col("revenue").desc())
)
display(top_drivers.toPandas())

,driver_id,driver_name,driver_experience,driver_rating,trips,revenue,avg_distance
0,cad5e297-e6b1-4665-8699-70498e2eac2a,Robert Smith,20,4.79,33,1735.00,15.43
1,a5c2948b-eed6-4fbb-a8a8-2a070c425b3f,George Morales,15,4.25,27,1594.91,17.83
2,b064fbfd-1b31-48ca-bb0a-2baa27810664,Alicia Lopez,23,4.68,26,1463.02,15.54
3,f452cbd8-63fa-421d-8c1f-1c4fff811cc3,Steven Stevens,16,4.26,23,1384.18,17.64
4,fff569c2-ee2b-48e4-93d1-622569a81137,Jenna Roman,1,4.88,26,1369.31,15.10
5,1c7d03db-dcfa-4897-a030-1c524d00fd4a,John Rodriguez,10,4.22,22,1322.84,16.74
6,6e149b07-7559-4c0e-8dbb-1cb464be22a3,Shelby Perry,15,4.69,21,1274.03,19.41
7,2e3ee8c4-1fc4-43ca-a371-db42a606e864,Amanda Miller,8,4.75,20,1274.01,17.64
8,258d5a93-958e-4aa7-8dfe-519dc24c472e,Jennifer Thompson,11,4.94,21,1267.66,17.15
9,71fad878-b369-4102-89f3-93d78b8ae077,Rachel Johnson,17,4.65,23,1243.83,15.01


---

## ✅ Checkpoint

- [x] Synthetic `driver_id` shared by trips and driver profiles
- [x] `LEFT JOIN` of `silver.taxi` with `silver.drivers`
- [x] `gold.enriched_trips` written; no trip lost, match rate reported

> **Next up ▶ [7 · Refactor via Delta Live Tables](./7.%20Refactor%20via%20Delta%20Live%20Tables.ipynb)** — recreate the whole pipeline declaratively with Delta Live Tables.

<div align="center"><sub>Yandex Go Analytics · notebook 6 of 8 · <a href="https://github.com/stanislavpanfilenko/YandexGoAnalytics">GitHub</a>